In [1]:
import pandas as pd
import numpy as np
import uuid

# 1. Configuration & Reproducibility
RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)

N_ASSETS = 5000
N_EVENTS = 20

print(f"Initializing generator with {N_ASSETS} assets and {N_EVENTS} hazard events...")

Initializing generator with 5000 assets and 20 hazard events...


In [2]:
def generate_assets(n_assets):
    asset_types = ['School', 'Hospital', 'Bridge', 'Power Substation', 'Water Pump']
    regions = ['NCR', 'Region III', 'Region IV-A', 'Region V', 'Region VIII']

    # Generate base features using numpy for speed
    df = pd.DataFrame({
        'asset_id': [f"AST-{str(uuid.uuid4())[:8]}" for _ in range(n_assets)],
        'asset_type': np.random.choice(asset_types, n_assets, p=[0.4, 0.1, 0.2, 0.15, 0.15]),
        'region': np.random.choice(regions, n_assets),
        'year_built': np.random.randint(1980, 2024, size=n_assets),
        'elevation_m': np.random.exponential(scale=15.0, size=n_assets) # Most are low elevation, some high
    })

    # REALISM INJECTION 1: Missingness
    # 10% of assets have lost their year_built records
    missing_mask = np.random.rand(n_assets) < 0.10
    df.loc[missing_mask, 'year_built'] = np.nan

    # THE OMITTED VARIABLE: True Maintenance Deficit (0 = perfect, 1 = severely neglected)
    # Using a Beta distribution skewed slightly towards good maintenance
    df['true_maintenance_deficit'] = np.random.beta(a=2, b=5, size=n_assets)

    # Official Vulnerability Score is correlated with true deficit, but flawed
    noise = np.random.normal(0, 0.15, size=n_assets)
    df['official_vulnerability_score'] = np.clip(df['true_maintenance_deficit'] - 0.1 + noise, 0, 1)

    # 5% missingness in official scores
    df.loc[np.random.rand(n_assets) < 0.05, 'official_vulnerability_score'] = np.nan

    return df

df_assets = generate_assets(N_ASSETS)
print(f"Assets generated: {df_assets.shape}")

Assets generated: (5000, 7)


In [3]:
def generate_hazard_events(n_events):
    event_types = ['Typhoon', 'Severe Flood', 'Monsoon']

    df = pd.DataFrame({
        'event_id': [f"EVT-{2010+i}-{str(uuid.uuid4())[:4]}" for i in range(n_events)],
        'event_type': np.random.choice(event_types, n_events, p=[0.6, 0.3, 0.1]),
        'peak_wind_speed_kph': np.random.uniform(60, 280, size=n_events),
        'flood_depth_mm': np.random.uniform(0, 1500, size=n_events),
        'duration_hours': np.random.randint(12, 72, size=n_events)
    })

    # Logical override: Monsoons don't have 280kph winds
    df.loc[df['event_type'] == 'Monsoon', 'peak_wind_speed_kph'] = np.random.uniform(40, 80, sum(df['event_type'] == 'Monsoon'))

    return df

df_events = generate_hazard_events(N_EVENTS)
print(f"Hazard Events generated: {df_events.shape}")

Hazard Events generated: (20, 5)


In [4]:
def generate_damage_reports(df_a, df_e):
    # Cross join: Every asset experiences every event (for simplicity in this scaffold)
    # Note: In a real database, we'd only generate reports for assets in the affected region.
    df_a['_tmp'] = 1
    df_e['_tmp'] = 1
    df = pd.merge(df_a, df_e, on='_tmp').drop('_tmp', axis=1)

    # 1. Normalize Hazards
    w_norm = (df['peak_wind_speed_kph'] - 60) / 240
    w_norm = np.clip(w_norm, 0, 1)

    # Effective flood (Water depth above elevation)
    eff_flood = np.maximum(0, df['flood_depth_mm'] - (df['elevation_m'] * 1000)) # convert m to mm
    f_norm = np.clip(eff_flood / 2000, 0, 1)

    # Impute missing official scores with median just for the math engine to run
    v_off = df['official_vulnerability_score'].fillna(df['official_vulnerability_score'].median())
    m_def = df['true_maintenance_deficit']

    # 2. The Stress Equation (Z) - Using our agreed upon Beta coefficients
    b0, b1, b2, b3, b4, b5 = -2.5, 3.0, 3.5, 2.0, 1.0, 1.5
    noise = np.random.normal(0, 0.5, size=len(df))

    Z = b0 + (b1 * w_norm) + (b2 * f_norm) + (b3 * v_off) + (b4 * m_def) + (b5 * w_norm * m_def) + noise

    # 3. Logistic S-Curve for Probability
    P_damage = 1 / (1 + np.exp(-Z))

    # 4. Binning into states (with minor random jitter)
    conditions = [
        (P_damage < 0.20),
        (P_damage >= 0.20) & (P_damage < 0.50),
        (P_damage >= 0.50) & (P_damage < 0.75),
        (P_damage >= 0.75) & (P_damage < 0.90),
        (P_damage >= 0.90)
    ]
    choices = ['No Damage', 'Minor', 'Moderate', 'Severe', 'Destroyed']
    df['damage_state'] = np.select(conditions, choices, default='No Damage')

    # 5. Service Disruption Logic
    df['is_service_disrupted'] = np.where(df['damage_state'].isin(['Severe', 'Destroyed']), 1,
                                          np.where(df['damage_state'] == 'Moderate', np.random.choice([0, 1], len(df), p=[0.7, 0.3]), 0))

    # Clean up columns to match schema
    df['report_id'] = [f"REP-{i}" for i in range(len(df))]
    columns_to_keep = ['report_id', 'asset_id', 'event_id', 'damage_state', 'is_service_disrupted', 'true_maintenance_deficit']
    return df[columns_to_keep]

df_damage = generate_damage_reports(df_assets.copy(), df_events.copy())
print(f"Damage Reports generated: {df_damage.shape}")

Damage Reports generated: (100000, 6)
